# Kaggle: hotword toàn cục + RNNLM (LM fusion) + đánh giá VAD cho Zipformer

Chạy phần nặng trên Kaggle (GPU cho RNNLM). Kết quả tải về máy để **suy luận offline** (không đọc `state.json`, không gọi mạng):

| Bước | Script | Output |
|---|---|---|
| A. Decode train **không hotword** (beam) | `eval_asr.py --configs beam --dump` | `asr_train_plain.jsonl` |
| B. Khai thác hotword từ lỗi ASR (chỉ train-split, id % 10 != 9) | `mine_hotwords.py` | `hotwords_global.txt/.json` |
| C. Train RNNLM trên token BPE của Zipformer | `train_rnnlm.py` | `rnnlm/rnnlm.onnx` |
| D. So WER trên held-out (id % 10 == 9) | `eval_asr.py` | bảng WER → chọn cấu hình |

**Chuẩn bị (Add Input):**
1. Dataset dữ liệu BTC (thư mục chứa `train/`), ví dụ `/kaggle/input/zaic2026/train+public_test`.
2. Dataset code: upload các thư mục `asr/ hotwords/ lm/ textnorm/ mapper/` của repo (giữ nguyên cấu trúc, có `__init__.py`).
3. Dataset model: thư mục snapshot `hynt/Zipformer-30M-RNNT-6000h` (`encoder/decoder/joiner-*.onnx`, `bpe.model`), lấy từ `~/.cache/huggingface/hub/models--hynt--Zipformer-30M-RNNT-6000h/snapshots/<hash>/`.

Bật **GPU** (cho bước C) và **Internet** (chỉ để `pip install`; hoặc thêm dataset wheel rồi cài `--no-index`).

## 1. Cấu hình

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess

ROOT      = Path("/kaggle/input/zaic2026/train+public_test")   # chứa train/train.jsonl, train/audio/
CODE_IN   = Path("/kaggle/input/voice-agent-code")              # chứa asr/ hotwords/ lm/ textnorm/ mapper/
MODEL_IN  = Path("/kaggle/input/zipformer-30m-rnnt-6000h")      # encoder/decoder/joiner-*.onnx + bpe.model
WORK      = Path("/kaggle/working")
MODEL_DIR = WORK / "zipformer"                                    # bản copy ghi được (cần tokens.txt / bpe.vocab)
THREADS   = os.cpu_count()

assert (ROOT / "train" / "train.jsonl").exists(), ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
print("cpu", THREADS)

## 2. Cài đặt

In [ ]:
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("sherpa-onnx==2.0.1", "sentencepiece", "soundfile", "silero-vad", "onnx", "onnxruntime")
# không có Internet: pip("--no-index", "--find-links", "/kaggle/input/<wheels>", "sherpa-onnx", ...)
import torch; print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 3. Code + model

In [ ]:
for d in ("asr", "hotwords", "lm", "textnorm", "mapper"):
    shutil.copytree(CODE_IN / d, WORK / d, dirs_exist_ok=True)
(WORK / "data").mkdir(exist_ok=True)
if not MODEL_DIR.exists():
    shutil.copytree(MODEL_IN, MODEL_DIR)
os.chdir(WORK); sys.path.insert(0, str(WORK))
from asr.asr_zipformer import _ensure_tokens
_ensure_tokens(MODEL_DIR)                       # tạo tokens.txt / bpe.vocab nếu thiếu
print(sorted(p.name for p in MODEL_DIR.iterdir()))
!python textnorm/vi_num.py && python asr/asr_fix.py

## A. Decode train không hotword → `asr_train_plain.jsonl`

`asr_train.jsonl` cũ được tạo với `--bias` (hotword + snap từ `state.json`) nên không dùng để khai thác. ~5000 câu, CPU.

In [ ]:
!python asr/eval_asr.py --root {ROOT} --subset all --configs beam --zipformer {MODEL_DIR} --threads {THREADS} \
    --bs 64 --dump data/asr_train_plain.jsonl | tail -4

## B. Khai thác hotword toàn cục (chỉ train-split)

In [ ]:
!python hotwords/mine_hotwords.py --in data/asr_train_plain.jsonl --train-jsonl {ROOT}/train/train.jsonl --zipformer {MODEL_DIR} \
    --min-err 2 --min-rate 0.3 --out hotwords/hotwords_global
!wc -l hotwords/hotwords_global.txt && head -40 hotwords/hotwords_global.txt

Duyệt nhanh: cụm nào bị nghe sai nhiều nhất, ASR nghe thành gì, alias dùng cho `snap_text`.

In [ ]:
import json, pandas as pd
rep = json.load(open("hotwords/hotwords_global.json", encoding="utf-8"))
pd.set_option("display.max_colwidth", 80)
pd.DataFrame([{k: r[k] for k in ("written", "kind", "hotwords", "score", "n_err", "n_occ", "err_rate")} |
              {"heard": [h for h, _ in r["heard"][:3]]} for r in rep]).head(60)

## C. Train RNNLM (GPU) → `rnnlm/rnnlm.onnx`

Corpus = `ref_spoken` của train-split (+ cách đọc số khác); held-out để early-stop. ONNX: `(x, x_lens) -> nll`, đúng
`OfflineRnnLM::Rescore` của sherpa-onnx. Có thể thử `--hidden 512 --emb 512` hoặc `--with-eos`.

In [ ]:
!python lm/train_rnnlm.py --data data/asr_train_plain.jsonl --zipformer {MODEL_DIR} --out lm/rnnlm \
    --emb 256 --hidden 256 --layers 2 --dropout 0.3 --epochs 40 --patience 4

In [ ]:
# kiểm tra sherpa-onnx nạp được LM + hotword
from asr.asr_zipformer import ZipformerASR
import glob
fs = sorted(glob.glob(str(ROOT / "train/audio/train_001*/train_001?.wav")))[:3]
for sc in (0.0, 0.3):
    asr = ZipformerASR(str(MODEL_DIR), num_threads=THREADS, bias=False, hotwords_file="hotwords/hotwords_global.txt",
                       lm="lm/rnnlm/rnnlm.onnx" if sc else None, lm_scale=sc)
    print(sc, asr.transcribe(fs))

## D. So sánh trên held-out (id % 10 == 9 — không dùng để khai thác hotword / train LM)

Ref = `audio_text` đổi sang dạng đọc (chấp nhận mọi cách đọc số hợp lệ). `snap` = snap_text theo `hotwords_global.json`,
`fix` = `asr_fix`. Chú ý: sherpa-onnx offline dùng LM để chấm lại các giả thuyết trong beam → thử thêm beam 8.

In [ ]:
E = f"python asr/eval_asr.py --root {ROOT} --zipformer {MODEL_DIR} --threads {THREADS} --bs 64 --hotwords hotwords/hotwords_global.txt --lm lm/rnnlm/rnnlm.onnx"
!{E} --configs greedy,beam,hw,hw+snap+fix,hw+lm,hw+lm+snap+fix --lm-scales 0.1,0.2,0.3,0.5 --dump-all eval_beam4 | tee eval_beam4.txt

In [ ]:
!{E} --beam 8 --configs hw,hw+lm --lm-scales 0.2,0.3,0.5 | tee eval_beam8.txt

VAD: bản mặc định (pad 0.15, rút khoảng lặng giữa > 0.5s) làm WER tệ hơn trên 64 câu thử ở local → dò pad / không rút khoảng lặng.

In [ ]:
for pad, gap in [(0.15, 0.5), (0.3, 99), (0.5, 99)]:
    print(f"== VAD pad={pad} gap={gap}")
    !{E} --configs hw,vad+hw --vad-pad {pad} --vad-gap {gap} | tail -3

## E. Tải kết quả về

Giải nén ở gốc repo local (đúng thư mục hotwords/, lm/, data/) rồi chạy offline, ví dụ:
```
python asr/test_asr_public.py --root DATA --hotwords-file hotwords/hotwords_global.txt --lm lm/rnnlm/rnnlm.onnx --lm-scale 0.3 [--vad]
python pipeline.py --root DATA --split public_test --asr-model zipformer --hotwords-file hotwords/hotwords_global.txt --lm lm/rnnlm/rnnlm.onnx
```

In [ ]:
!zip -qr asr_assets.zip hotwords/hotwords_global.txt hotwords/hotwords_global.json lm/rnnlm/rnnlm.onnx data/asr_train_plain.jsonl eval_beam*.txt
!ls -la asr_assets.zip